<a href="https://colab.research.google.com/github/LOHITHVATTAM1628/Text-Summarization/blob/main/Text_Summarizer_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Jul  2 17:25:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
pip install transformers

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)


In [ ]:
!pip install -q evaluate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset,load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
from datasets import load_dataset, load_from_disk
import evaluate
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import torch

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
model_ckpt="google/pegasus-cnn_dailymail"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
!pip install -U datasets

In [ ]:
dataset_samsum = load_dataset("knkarthick/samsum")

In [ ]:
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [ ]:
dataset_samsum["train"]["dialogue"][1]

'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great'

In [ ]:
dataset_samsum["train"]["summary"][1]

'Olivia and Olivier are voting for liberals in this election. '

In [ ]:
split_lengths=[len(dataset_samsum[split])for split in dataset_samsum]
print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")
print(dataset_samsum["test"][1]["dialogue"])
print("\nSummary:")
print(dataset_samsum["test"][1]["summary"])

Split lengths: [14731, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary:
Eric and Rob are going to watch a stand-up on youtube.


In [ ]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["dialogue"],
        max_length=1024,
        truncation=True
    )

    target_encodings = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [ ]:
dataset_samsum_pt=dataset_samsum.map(convert_examples_to_features,batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [ ]:
dataset_samsum_pt["train"]["input_ids"][1]

[18038,
 151,
 2632,
 127,
 119,
 6228,
 118,
 115,
 136,
 2974,
 152,
 10463,
 151,
 35884,
 130,
 329,
 107,
 18038,
 151,
 2587,
 314,
 1242,
 10463,
 151,
 1509,
 1]

In [ ]:
dataset_samsum_pt["train"][1]["attention_mask"]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [ ]:
#Training
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",   # <-- changed
    eval_steps=500,
    save_steps=1000000,
    gradient_accumulation_steps=16,
)

In [ ]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
52,46.891028,2.416516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=48.956233318035416, metrics={'train_runtime': 444.8618, 'train_samples_per_second': 1.841, 'train_steps_per_second': 0.117, 'total_flos': 322454634700800.0, 'train_loss': 48.956233318035416, 'epoch': 1.0})

In [ ]:
import evaluate
import numpy as np
from tqdm import tqdm

rouge = evaluate.load("rouge")

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Yield successive batch-sized chunks."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]


def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=8,
    device="cuda"
):
    article_batches = list(
        generate_batch_sized_chunks(dataset["dialogue"], batch_size)
    )

    target_batches = list(
        generate_batch_sized_chunks(dataset["summary"], batch_size)
    )

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches),
        total=len(article_batches)
    ):

        inputs = tokenizer(
            article_batch,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        summaries = model.generate(
            **inputs,
            max_length=64
        )

        decoded_summaries = tokenizer.batch_decode(
            summaries,
            skip_special_tokens=True
        )

        metric.add_batch(
            predictions=decoded_summaries,
            references=target_batch
        )

    return metric.compute()

In [ ]:
import evaluate
import torch
from tqdm import tqdm

# Load ROUGE metric
rouge = evaluate.load("rouge")

def generate_batch_sized_chunks(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]

def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=2,
    device="cuda"
):
    # Take only first 10 examples
    dialogues = dataset["dialogue"][:10]
    summaries = dataset["summary"][:10]

    article_batches = list(generate_batch_sized_chunks(dialogues, batch_size))
    target_batches = list(generate_batch_sized_chunks(summaries, batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches),
        total=len(article_batches)
    ):
        inputs = tokenizer(
            article_batch,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        summaries_ids = model.generate(
            **inputs,
            max_length=64
        )

        decoded_summaries = tokenizer.batch_decode(
            summaries_ids,
            skip_special_tokens=True
        )

        metric.add_batch(
            predictions=decoded_summaries,
            references=target_batch
        )

    return metric.compute()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

score = calculate_metric_on_test_ds(
    dataset_samsum["test"],
    rouge,
    model_pegasus,
    tokenizer,
    batch_size=2,
    device=device
)

print(score)

100%|██████████| 5/5 [00:12<00:00,  2.53s/it]

{'rouge1': np.float64(0.337527420158949), 'rouge2': np.float64(0.09081476465246233), 'rougeL': np.float64(0.25765663065701194), 'rougeLsum': np.float64(0.26115668964954697)}


In [ ]:
print(score)

{'rouge1': np.float64(0.337527420158949), 'rouge2': np.float64(0.09081476465246233), 'rougeL': np.float64(0.25765663065701194), 'rougeLsum': np.float64(0.26115668964954697)}


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_pegasus.to(device)

for i in range(10):
    dialogue = dataset_samsum["test"][i]["dialogue"]

    inputs = tokenizer(
        dialogue,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model_pegasus.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    predicted_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    print("=" * 100)
    print(f"Example {i+1}")
    print("\nDialogue:\n")
    print(dialogue)

    print("\nOriginal Summary:\n")
    print(dataset_samsum["test"][i]["summary"])

    print("\nPredicted Summary:\n")
    print(predicted_summary)
    print("=" * 100)

Example 1

Dialogue:

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Original Summary:

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Predicted Summary:

Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him .
Example 2

Dialogue:

Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: S